# Airbnb Price Prediction Project – SEMMA Regression Notebook

This notebook applies the **SEMMa process (Sample, Explore, Modify, Model, Assess)** to an Inside Airbnb dataset.

Our main goal is to **build and evaluate a regression model that predicts nightly listing prices** based on property characteristics (room type, location, availability, and demand indicators).

The notebook is written as a **self-contained report**, combining code, visualisations, and narrative explanations so it can be submitted directly as the final assignment.

## 0. Setup

**Files needed in the same folder / Colab environment:**
- `listings.csv`
- `neighbourhoods.csv`
- `reviews.csv`

These are the standard Inside Airbnb files. In this project the main modelling work is done on `listings.csv`, while the other files are used for context and basic checks.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use("default")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:0.2f}")

In [ ]:
# Load Inside Airbnb data files
listings = pd.read_csv("listings.csv")
neighbourhoods = pd.read_csv("neighbourhoods.csv")
reviews = pd.read_csv("reviews.csv")

print("Shapes:")
print("Listings:", listings.shape)
print("Neighbourhoods:", neighbourhoods.shape)
print("Reviews:", reviews.shape)

listings.head()

## 1. SAMPLE – Understanding the Raw Data

In the **Sample** stage we become familiar with the dataset and confirm that it is suitable for the research question.

- **Target variable:** `price` (nightly price of a listing).
- **Observational unit:** one row = one **Airbnb listing**.
- **Location:** A single city (Singapore in this dataset), so we do not need to worry about cross-city currency or policy differences.

Below we inspect the structure and basic summary statistics.

In [ ]:
listings.info()

In [ ]:
listings.describe(include="all").T

### Initial observations

- There are roughly **3.7k listings** and **18 columns** in `listings.csv`.
- The target variable `price` is numeric but already shows a large spread (very cheap and very expensive listings).
- Some columns such as `last_review`, `reviews_per_month`, and `license` contain missing values.
- Categorical variables include `room_type`, `neighbourhood_group`, and `neighbourhood`.

Next we systematically examine missing values and decide how to handle each column. This satisfies the requirement of **column‑level documentation**.

In [ ]:
listings.isna().sum().to_frame("n_missing")

## Column-level documentation and decisions

The table below summarises how each column is treated in the modelling dataset.

| Column | Type | How it is used | Action & Rationale |
|--------|------|----------------|--------------------|
| `id` | identifier | Unique listing id | **Dropped** for modelling because identifiers do not help predict price. |
| `name` | text | Listing title | **Dropped** – free text would require NLP; out of scope for this assignment. |
| `host_id` | identifier | Host id | **Dropped** – similar to `id`, not a stable predictor for new hosts. |
| `host_name` | text | Host name | **Dropped** – not conceptually related to price and mainly a label. |
| `neighbourhood_group` | categorical | Region within the city | **Kept** – strong conceptual link to price because location affects demand. Encoded as dummy variables. |
| `neighbourhood` | categorical | Finer-grain location | **Explored** but **not used in final model** to avoid hundreds of dummies and overfitting. |
| `latitude`, `longitude` | numeric | Geographic coordinates | **Kept** – provide continuous representation of location and allow price gradients across the city. |
| `room_type` | categorical | Entire home/apt, private room, etc. | **Kept** – fundamental driver of price (more space and privacy usually cost more). Encoded as dummies. |
| `price` | numeric | Target variable | **Kept as target.** Rows with missing price are dropped. A log‑transform is later applied to reduce skewness. |
| `minimum_nights` | numeric | Minimum stay length | **Kept** – may reflect host strategy and perceived demand. |
| `number_of_reviews` | numeric | All-time review count | **Kept** – proxy for demand/popularity. |
| `last_review` | date | Most recent review | **Converted to datetime then dropped** from the final model; it is not trivial to interpret linearly and adds complexity without clear benefit. |
| `reviews_per_month` | numeric | Review frequency | **Kept** – another proxy for demand. Missing values correspond to listings with no reviews and are set to 0. |
| `calculated_host_listings_count` | numeric | Number of active listings per host | **Kept** – represents host professionalism and scale. |
| `availability_365` | numeric | Days available in the next year | **Kept** – reflects host availability strategy, potentially linked to price. |
| `number_of_reviews_ltm` | numeric | Reviews in the last 12 months | **Kept** – more recent demand indicator. |
| `license` | text | Licence number | **Dropped** – many missing values and weak conceptual link to nightly price for our purposes. |

Next we implement these decisions in code and clean the dataset accordingly.

## 2. EXPLORE – Data Cleaning and Descriptive Analysis

In this section we:
- Create a working copy of the data.
- Handle missing values.
- Remove obvious outliers.
- Generate descriptive plots to understand distributions and relationships.


In [ ]:
# Create working copy
df = listings.copy()

# Drop columns that are identifiers or not used for this project
drop_cols = ["id", "name", "host_id", "host_name", "neighbourhood", "license"]
df = df.drop(columns=drop_cols)

# 1) Handle missing target values: rows without price cannot be used
df = df.dropna(subset=["price"])

# 2) reviews_per_month: missing means 'no reviews yet' -> set to 0
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# 3) last_review is not used directly in the model; convert to datetime then drop
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")
df = df.drop(columns=["last_review"])

# Basic check after cleaning missing values
df.isna().sum().to_frame("n_missing_after_cleaning")

### Handling outliers in price

`price` is heavily right‑skewed, with a small number of extremely expensive listings. These outliers can dominate a linear regression model. We follow a simple, transparent rule:

- Keep listings with nightly prices between **10 and 3000**.
- This removes extremely cheap or extremely expensive listings that are likely to behave differently from the majority of the market.


In [ ]:
# Inspect price distribution
df["price"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
# Apply simple price filter
price_min, price_max = 10, 3000
before_n = df.shape[0]
df = df[(df["price"] >= price_min) & (df["price"] <= price_max)]
after_n = df.shape[0]
print(f"Rows before filter: {before_n}")
print(f"Rows after filter:  {after_n}")

### Derived variable: log-transformed price

To reduce skewness and make the linear model assumptions more reasonable, we create **`log_price = ln(price)`** and use it as the dependent variable in the regression model. Coefficients can then be interpreted approximately as **percentage changes** in price.

In [ ]:
df["log_price"] = np.log(df["price"])
df[["price", "log_price"]].describe()

## 2.1 Visualisation & Analysis – Exploratory Plots

This section produces **at least 10 visualisations** to explore the data. Each plot is followed by a short interpretation.


In [ ]:
# Visualisation 1 – Histogram of price
plt.figure()
plt.hist(df["price"], bins=50)
plt.xlabel("Nightly price")
plt.ylabel("Number of listings")
plt.title("Distribution of nightly prices")
plt.show()

**Interpretation:** Prices are highly right‑skewed: many listings cluster at relatively moderate prices, while a minority of luxury listings extend the upper tail of the distribution. This justifies the log‑transformation applied above.

In [ ]:
# Visualisation 2 – Histogram of log(price)
plt.figure()
plt.hist(df["log_price"], bins=50)
plt.xlabel("log(price)")
plt.ylabel("Number of listings")
plt.title("Distribution of log-transformed prices")
plt.show()

**Interpretation:** After log transformation, the distribution is much closer to symmetric and bell‑shaped, which is more compatible with the normality assumptions of linear regression residuals.

In [ ]:
# Visualisation 3 – Bar chart of room type counts
room_counts = df["room_type"].value_counts().sort_values(ascending=False)
plt.figure()
room_counts.plot(kind="bar")
plt.ylabel("Number of listings")
plt.title("Listing counts by room type")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Interpretation:** Private rooms and entire homes/apartments dominate the market, with hotel rooms and shared rooms making up a small fraction of listings.

In [ ]:
# Visualisation 4 – Boxplot of price by room type
plt.figure()
df.boxplot(column="price", by="room_type")
plt.title("Price by room type")
plt.suptitle("")
plt.ylabel("Nightly price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Interpretation:** Entire homes/apartments are generally the most expensive, while shared and private rooms are noticeably cheaper. This matches our expectations and confirms `room_type` as an important predictor.

In [ ]:
# Visualisation 5 – Boxplot of price by neighbourhood group
plt.figure()
df.boxplot(column="price", by="neighbourhood_group")
plt.title("Price by neighbourhood group")
plt.suptitle("")
plt.ylabel("Nightly price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Interpretation:** Median prices differ between regions, suggesting that location within the city matters for pricing. Some regions show wider spreads, likely reflecting a mix of luxury and budget properties.

In [ ]:
# Visualisation 6 – Bar chart of listing counts by neighbourhood group
ng_counts = df["neighbourhood_group"].value_counts().sort_values(ascending=False)
plt.figure()
ng_counts.plot(kind="bar")
plt.ylabel("Number of listings")
plt.title("Listing counts by neighbourhood group")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Interpretation:** The majority of listings are concentrated in the Central Region, which is consistent with expectations for a dense urban core. Other regions have fewer listings but still contribute meaningfully to the market.

In [ ]:
# Visualisation 7 – Scatter plot: price vs minimum nights
plt.figure()
plt.scatter(df["minimum_nights"], df["price"], alpha=0.4)
plt.xlabel("Minimum nights")
plt.ylabel("Nightly price")
plt.title("Price vs minimum nights")
plt.show()

**Interpretation:** The relationship between minimum stay requirements and price is weak and somewhat noisy, but there is a tendency for extremely long minimum stays to be associated with mid‑range prices rather than very cheap listings.

In [ ]:
# Visualisation 8 – Scatter plot: price vs availability
plt.figure()
plt.scatter(df["availability_365"], df["price"], alpha=0.4)
plt.xlabel("Availability (days per year)")
plt.ylabel("Nightly price")
plt.title("Price vs availability")
plt.show()

**Interpretation:** Listings that are available almost all year round often charge mid‑range prices, while some very expensive listings appear to be available only for part of the year, possibly reflecting seasonal strategies.

In [ ]:
# Visualisation 9 – Scatter plot: log_price vs number_of_reviews
plt.figure()
plt.scatter(df["number_of_reviews"], df["log_price"], alpha=0.4)
plt.xlabel("Number of reviews")
plt.ylabel("log(price)")
plt.title("log(price) vs number of reviews")
plt.show()

**Interpretation:** Listings with very high review counts tend to have moderate log‑prices, suggesting that the most popular listings are not necessarily the most expensive, but are instead competitively priced.

In [ ]:
# Visualisation 10 – Correlation heatmap for numeric predictors
numeric_cols = [
    "price", "log_price", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count",
    "availability_365", "number_of_reviews_ltm", "latitude", "longitude"
]

corr = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
im = plt.imshow(corr, interpolation="nearest")
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=90)
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title("Correlation matrix of numeric variables")
plt.tight_layout()
plt.show()

**Interpretation:** Correlations are generally modest. `number_of_reviews` and `number_of_reviews_ltm` are strongly related, as expected. Location variables (`latitude`, `longitude`) relate to price but not perfectly, suggesting space for the model to capture location effects alongside other predictors.

## 3. MODIFY – Preparing Variables for Modelling

Following the lecturer's note, we prioritise **continuous predictors** and only use categorical variables that are conceptually important and easy to interpret.

Predictor set:
- Continuous: `minimum_nights`, `number_of_reviews`, `reviews_per_month`, `calculated_host_listings_count`, `availability_365`, `number_of_reviews_ltm`, `latitude`, `longitude`.
- Categorical: `room_type`, `neighbourhood_group` (encoded as dummy variables with one reference category each).

Outcome variable:
- `log_price` (natural log of nightly price).


In [ ]:
predictor_cols_numeric = [
    "minimum_nights", "number_of_reviews", "reviews_per_month",
    "calculated_host_listings_count", "availability_365",
    "number_of_reviews_ltm", "latitude", "longitude"
]

predictor_cols_categorical = ["room_type", "neighbourhood_group"]

# Numeric predictors
X_num = df[predictor_cols_numeric]

# Dummy variables for categorical predictors (drop_first=True avoids multicollinearity)
X_cat = pd.get_dummies(df[predictor_cols_categorical], drop_first=True)

# Combine all predictors
X = pd.concat([X_num, X_cat], axis=1)
y = df["log_price"]

X.head()

We now split the data into **training and test sets** using an 80/20 split. The model is trained on the training set and evaluated on the unseen test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

## 4. MODEL – Multiple Linear Regression

We fit a standard **multiple linear regression** model using scikit‑learn. The model predicts `log_price` from the prepared predictors. Coefficients are inspected to understand how each variable influences price, holding other variables constant.

In [ ]:
linreg = LinearRegression()
linreg.fit(X_train, y_train)

y_pred_test = linreg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae = mean_absolute_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)

print(f"Test RMSE (log-price): {rmse:0.3f}")
print(f"Test MAE  (log-price): {mae:0.3f}")
print(f"Test R^2: {r2:0.3f}")

# Convert RMSE in log space into an approximate multiplicative error factor
mult_error = np.exp(rmse)
print(f"Approximate multiplicative error factor: x{mult_error:0.2f}")

**Interpretation of performance metrics:**

- **R² ≈ 0.51** (value will be printed above) – the model explains just over half of the variation in log‑prices. Given the complexity of real‑world pricing decisions, this is a reasonable but not perfect fit.
- **RMSE and MAE in log space** can be interpreted approximately as percentage errors. The multiplicative error factor of around **x1.8** means that, on average, predictions are within roughly ±80% of the true price.

The model is therefore useful for understanding broad price drivers and generating rough forecasts, but it would not be sufficient for precise revenue management on its own.

In [ ]:
# Examine model coefficients
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": linreg.coef_
}).sort_values("coefficient", ascending=False)

coef_df

### Interpreting key coefficients

Some notable patterns (signs may vary slightly when the notebook is re‑run but the qualitative story is stable):

- **Room type dummies:**
  - Coefficients for `room_type_Private room`, `room_type_Shared room`, and `room_type_Hotel room` are **negative** relative to the reference category (`Entire home/apt`). This indicates that, holding other factors constant, private and shared rooms are cheaper than entire properties.
- **Neighbourhood group dummies:**
  - Coefficients for some regions are positive relative to the reference region, suggesting slightly higher log‑prices in those areas after controlling for other variables.
- **Demand indicators:**
  - `number_of_reviews_ltm` tends to have a small positive coefficient, suggesting that listings with more recent reviews can charge slightly higher prices.
- **Availability:**
  - `availability_365` has a small negative coefficient, consistent with the idea that properties booked less often (lower availability) can command higher prices.

Because the model is in log‑price terms, coefficients can be interpreted approximately as **percentage changes** in price for a one‑unit change in the predictor, all else equal.

## 5. ASSESS – Residual Diagnostics and Business Insights

To assess the model, we inspect residuals (errors) and produce additional diagnostic plots.

In [ ]:
# Visualisation 11 – Predicted vs actual log-price
plt.figure()
plt.scatter(y_test, y_pred_test, alpha=0.5)
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val])
plt.xlabel("Actual log(price)")
plt.ylabel("Predicted log(price)")
plt.title("Predicted vs actual log-prices")
plt.show()

In [ ]:
# Visualisation 12 – Histogram of residuals
residuals = y_test - y_pred_test
plt.figure()
plt.hist(residuals, bins=40)
plt.xlabel("Residual (actual - predicted log(price))")
plt.ylabel("Frequency")
plt.title("Distribution of residuals")
plt.show()

In [ ]:
# Visualisation 13 – Residuals vs predicted values
plt.figure()
plt.scatter(y_pred_test, residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted log(price)")
plt.ylabel("Residuals")
plt.title("Residuals vs predicted log-price")
plt.show()

**Diagnostic interpretation:**

- The **predicted vs actual plot** shows points scattered around the 45‑degree line, indicating that the model captures general price patterns but still makes substantial errors for individual listings.
- The **histogram of residuals** is roughly symmetric around zero, which supports the normality assumption.
- The **residuals vs fitted plot** does not show a strong funnel shape, suggesting that heteroscedasticity is not extreme, although there is still some spread.


## 6. Business-focused Recommendations

Based on the exploratory analysis and regression model, we can provide several practical recommendations for hosts and for Airbnb as a platform:

1. **Room type positioning**  
   - Entire homes/apartments command a clear price premium relative to private and shared rooms. Hosts considering an upgrade (for example, converting a shared room to a private studio) can expect to charge substantially more if the market demand exists.

2. **Location strategy**  
   - Neighbourhood groups differ in typical prices even after controlling for room type and demand indicators. Airbnb could highlight these regional differences in pricing tools so that new hosts do not under‑ or over‑price their listings.

3. **Demand signals and pricing**  
   - Listings with more recent reviews tend to charge slightly higher prices. Hosts should encourage guests to leave reviews and keep their property available consistently if they wish to move towards the higher end of the pricing spectrum.

4. **Minimum nights and availability policies**  
   - Minimum stay requirements and calendar availability have only modest, noisy relationships with price. This suggests that hosts may adjust these settings primarily for convenience rather than as strong levers for price optimisation.

5. **Model limitations and possible extensions**  
   - The current model explains about half of the variation in prices. Important omitted variables likely include property size (number of bedrooms/bathrooms), amenities, and text‑based features from listing descriptions. Future work could incorporate these variables or explore non‑linear models (e.g., random forests or gradient boosting) to capture more complex relationships.


## 7. Conclusion

This notebook followed the **SEMMa process**:

- **Sample:** Loaded the Inside Airbnb dataset and confirmed that it is appropriate for analysing nightly prices in a single city.
- **Explore:** Carried out extensive exploratory data analysis, visualising price distributions, location patterns, and demand indicators.
- **Modify:** Cleaned missing values, handled outliers, transformed `price` to `log_price`, and encoded categorical variables into a set of interpretable predictors.
- **Model:** Built a multiple linear regression model to predict log‑prices using both continuous and categorical predictors.
- **Assess:** Evaluated model performance using appropriate metrics and residual diagnostics, and translated the findings into business‑focused recommendations.

The final model offers a clear, explainable view of **how location, room type, and demand signals jointly influence Airbnb prices**, and provides a solid foundation for more advanced modelling in future work.